# SmartCivic — Road Damage YOLOv8 Detection Training (GPU)
This notebook trains the YOLOv8 model for road defect detection (`Crack`, `Pothole`, `Surface Erosion`) on a free GPU (T4/V100).
- Follows the SmartCivic Master Implementation Plan.
- Saves checkpoints and exported ONNX models directly to Google Drive.

In [ ]:
# 1. Verify GPU availability and install Ultralytics
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime

In [ ]:
# 2. Optional: Mount Google Drive to preserve checkpoints
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/SmartCivic_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_DIR)

In [ ]:
# 3. Upload and extract datasets (or download workspace dataset zip)
# If using the zipped 'work' dataset from your local machine:
# from google.colab import files
# files.upload() # upload smartcivic_dataset_work.zip
!mkdir -p /content/dataset
# !unzip -q smartcivic_dataset_work.zip -d /content/dataset/

In [ ]:
# 4. Define data.yaml
data_yaml = '''
path: /content/dataset
train: train/images
val: val/images
test: test/images
nc: 3
names: ['Crack', 'Pothole', 'Surface Erosion']
'''
with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)

In [ ]:
# 5. Full GPU Fine-Tuning (50 epochs, imgsz=640, early stopping patience=10)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    device=0,
    project='/content/runs',
    name='smartcivic_yolov8n'
)

In [ ]:
# 6. Export best model to ONNX with dynamic=False (CPU friendly)
best_model_path = '/content/runs/smartcivic_yolov8n/weights/best.pt'
best_model = YOLO(best_model_path)
onnx_file = best_model.export(format='onnx', imgsz=640, opset=12, dynamic=False)
print('Exported ONNX file:', onnx_file)
!cp {onnx_file} /content/drive/MyDrive/SmartCivic_Models/model.onnx
!cp {best_model_path} /content/drive/MyDrive/SmartCivic_Models/best.pt
print('Saved model to Google Drive!')